# cjm-capability-primitives

> The dep-light primitives library for the cjm capability ecosystem — the shared cross-task result DTOs (data nouns) that tool capabilities emit and task adapters, workflow cores, and the composition layer all consume.

**Result nouns vs adapter machinery:** a single-task tool capability emits its task's typed data noun (e.g. `TranscriptionResult`) and depends on *only* that — the adapter (`TaskAdapter`, the required tool protocol, the cache/persistence helpers) lives in the per-task `cjm-<task>-adapter-interface` lib and is constructed *around* the tool, never imported *by* it. The capability-side counterpart to `cjm-context-graph-primitives`.

## Install

```bash
pip install cjm_capability_primitives
```

## Project Structure

```
nbs/
├── forced_alignment.ipynb  # Standardized word-level forced-alignment DTOs — the data noun forced-alignment tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.
├── media_processing.ipynb  # Standardized result DTOs for the media-processing task — the data nouns media-processing tool capabilities (ffmpeg today) emit and the multi-method task adapter / workflow cores consume, wire-registered so results cross the worker boundary typed.
├── monitoring.ipynb        # Standardized telemetry DTOs for the system-monitor capability — the data nouns a monitor tool capability emits (host CPU/RAM + aggregated GPU stats, and per-process GPU usage) and the substrate scheduler consumes for resource-derived admission + GPU subtree attribution.
├── source_separation.ipynb # Standardized result DTO for the source-separation (audio-preprocessing) task — the data noun source-separation tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.
├── transcription.ipynb     # Standardized result DTO for the transcription task — the data noun tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.
└── vad.ipynb               # Standardized result DTO for the voice-activity-detection task — the data noun VAD tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.
```

Total: 6 notebooks

## Module Dependencies

```mermaid
graph LR
    forced_alignment["forced_alignment<br/>Forced Alignment Result"]
    media_processing["media_processing<br/>Media Processing Results"]
    monitoring["monitoring<br/>System Monitoring DTOs"]
    source_separation["source_separation<br/>Source Separation Result"]
    transcription["transcription<br/>Transcription Result"]
    vad["vad<br/>VAD Result"]

```

No cross-module dependencies detected.

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### Forced Alignment Result (`forced_alignment.ipynb`)
> Standardized word-level forced-alignment DTOs — the data noun forced-alignment tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.forced_alignment import (
    ForcedAlignItem,
    ForcedAlignResult
)
```
#### Classes

```python
@dataclass
class ForcedAlignItem:
    "A single word-level alignment result."
    
    text: str  # The aligned word (punctuation typically stripped by model)
    start_time: float  # Start time in seconds
    end_time: float  # End time in seconds
```

```python
@dataclass
class ForcedAlignResult:
    "Standardized output for all forced alignment capabilities."
    
    items: List[ForcedAlignItem]  # Word-level alignments
    metadata: Dict[str, Any] = field(...)  # Capability-specific metadata
    
    def from_dict(
        "Reconstruct from a wire payload, re-typing nested items.

`items` holds typed `ForcedAlignItem` objects, so the substrate's typed
wire envelope (stage 2) reconstructs them host-side here rather than
leaving bare dicts (which would break attribute access like `it.text`)."
```


### Media Processing Results (`media_processing.ipynb`)
> Standardized result DTOs for the media-processing task — the data nouns media-processing tool capabilities (ffmpeg today) emit and the multi-method task adapter / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.media_processing import (
    MediaSegment,
    MediaArtifactResult,
    MediaSegmentationResult,
    MediaMetadata
)
```
#### Classes

```python
@dataclass
class MediaSegment:
    """
    One produced segment file from a `segment_audio` batch cut.
    
    The per-segment entry the fused-era ffmpeg `segment_audio` returned as a
    dict, now a typed noun (the dead `job_id` dropped — born-final; the adapter
    owns persistence). Workflow cores read `index`/`output_path`/`start`/`end`
    to build the per-segment composition.
    """
    
    index: int  # 0-based position of this segment within the batch
    output_path: str  # Path to the produced segment file the tool wrote
    start: float  # Segment start time in the source (seconds)
    end: float  # Segment end time in the source (seconds)
    duration: float  # end - start (seconds)
    
    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        "Convert to dictionary for JSON serialization."
```

```python
@dataclass
class MediaArtifactResult:
    """
    A single produced audio artifact (the `convert` / `extract_audio` output).
    
    The artifact-producing shape (cf. `SourceSeparationResult`): `output_path`
    is the file the tool wrote to the adapter-chosen location; `metadata`
    carries the stats the fused-era return dict / row held (codec, duration,
    stream_copy, the effective convert parameters, ...). Flat fields (str +
    dict), so the default wire reconstruction suffices — no custom from_dict.
    """
    
    output_path: str  # Path to the produced audio file
    metadata: Dict[str, Any] = field(...)  # Stats (codec, duration, parameters, ...)
```

```python
@dataclass
class MediaSegmentationResult:
    """
    A BATCH of produced segment files (the `segment_audio` output).
    
    Holds typed `MediaSegment`s plus the batch metadata the fused-era return
    dict carried (`input_path`, `segment_count`, `total_duration`, `batch_key`
    — the label linking the cut files in the run manifest). Because `segments`
    holds typed objects, a custom `from_dict` re-types them on wire-decode (the
    auto flat reconstruct would leave bare dicts, breaking `seg.output_path`
    access) — the `VADResult` precedent.
    """
    
    segments: List[MediaSegment]  # The produced segment files, ordered by index
    input_path: str = ''  # The source audio that was cut
    segment_count: int = 0  # Number of segments produced
    total_duration: float = 0.0  # Sum of segment durations (seconds)
    batch_key: str = ''  # Label linking this batch's cut files (run-manifest field)
    
    def from_dict(
        "Reconstruct from a wire payload, re-typing nested MediaSegments."
```

```python
@dataclass
class MediaMetadata:
    """
    Probed metadata for a media file (the `get_info` result) — inline data, no artifact.
    
    Relocated to `cjm-capability-primitives` from the dissolving
    `cjm-media-plugin-system.core` (the `TranscriptionResult`/`ForcedAlignResult`
    relocation precedent). `get_info` is the media-processing task's UNCACHED
    probe op, so this is a read result, not a produced-artifact pointer. The
    stream lists are plain dicts, so the default wire reconstruction suffices.
    """
    
    path: str  # File path probed
    duration: float  # Duration in seconds
    format: str  # Container format (e.g. 'mp4', 'mkv')
    size_bytes: int  # File size in bytes
    video_streams: List[Dict[str, Any]] = field(...)  # Per-video-stream info (codec, width, height, fps)
    audio_streams: List[Dict[str, Any]] = field(...)  # Per-audio-stream info (codec, sample_rate, channels, duration)
    
    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        "Convert to dictionary for JSON serialization."
```


### System Monitoring DTOs (`monitoring.ipynb`)
> Standardized telemetry DTOs for the system-monitor capability — the data nouns a monitor tool capability emits (host CPU/RAM + aggregated GPU stats, and per-process GPU usage) and the substrate scheduler consumes for resource-derived admission + GPU subtree attribution.

#### Import

```python
from cjm_capability_primitives.monitoring import (
    SystemStats,
    ProcessStats,
    MonitorToolProtocol
)
```
#### Classes

```python
@dataclass
class SystemStats:
    "Standardized snapshot of host + GPU resources (the scheduler's admission input)."
    
    cpu_percent: float = 0.0  # Overall CPU utilization percentage
    memory_used_mb: float = 0.0  # Currently used system RAM in MB
    memory_total_mb: float = 0.0  # Total system RAM in MB
    memory_available_mb: float = 0.0  # Available system RAM in MB
    gpu_type: str = 'None'  # GPU vendor: 'NVIDIA', 'AMD', 'Intel', 'None'
    gpu_free_memory_mb: float = 0.0  # Free GPU memory in MB
    gpu_total_memory_mb: float = 0.0  # Total GPU memory in MB
    gpu_used_memory_mb: float = 0.0  # Used GPU memory in MB
    gpu_load_percent: float = 0.0  # GPU compute utilization percentage
    
    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        "Convert to dictionary for JSON serialization."
```

```python
@dataclass
class ProcessStats:
    "Per-process GPU usage, reported by a monitor's `list_processes` (GPU subtree attribution)."
    
    pid: int = 0  # OS process ID
    gpu_index: int = -1  # GPU index (0-based); -1 if not GPU-bound or unknown
    gpu_memory_mb: float = 0.0  # GPU memory attributable to this process, in MB
    command: str = ''  # Process command line (or short name)
    
    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        "Convert to dictionary for JSON serialization."
```

```python
@runtime_checkable
class MonitorToolProtocol(Protocol):
    """
    The native surface a system-monitor tool capability exposes.
    
    The substrate consumes a monitor through this surface by NAME (duck-typed,
    host-no-imports per CR-1) — `get_system_status` feeds resource-derived
    admission; `list_processes` feeds per-worker GPU subtree attribution. There
    is no task adapter: this is the native-dispatch contract, and the manifest's
    `structural_surface` is what a future auto-detect could match against. Platform
    monitors (NVIDIA today; Intel / AMD / Apple Silicon later) each implement it.
    """
    
    def get_system_status(self) -> SystemStats: ...      # Current host + aggregated GPU telemetry
        def list_processes(self) -> List[ProcessStats]: ...  # Per-process GPU usage ([] if no per-process visibility)
    
    def list_processes(self) -> List[ProcessStats]: ...  # Per-process GPU usage ([] if no per-process visibility)
```


### Source Separation Result (`source_separation.ipynb`)
> Standardized result DTO for the source-separation (audio-preprocessing) task — the data noun source-separation tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.source_separation import (
    SourceSeparationResult
)
```
#### Classes

```python
@dataclass
class SourceSeparationResult:
    """
    Standardized output for source-separation (audio-preprocessing) capabilities.
    
    The payload is an AUDIO ARTIFACT, not inline data: `output_path` is the
    produced isolated-audio file (e.g. the vocals stem) the tool wrote to the
    location the adapter chose. `metadata` carries the stats the fused-era
    return dict held (duration, sample_rate, model, stems_available, and any
    extra-stem paths when the tool was asked to keep them).
    """
    
    output_path: str  # Path to the produced isolated-audio artifact (e.g. vocals stem)
    metadata: Dict[str, Any] = field(...)  # Stats (duration, sample_rate, model, stems_available, other_stems, ...)
```


### Transcription Result (`transcription.ipynb`)
> Standardized result DTO for the transcription task — the data noun tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.transcription import (
    TranscriptionResult
)
```
#### Classes

```python
@dataclass
class TranscriptionResult:
    "Standardized output for all transcription plugins."
    
    text: str  # The transcribed text
    confidence: Optional[float]  # Overall confidence (0.0 to 1.0)
    segments: Optional[List[Dict[str, Any]]]  # Timestamped segments
    metadata: Dict[str, Any] = field(...)  # Additional metadata
```


### VAD Result (`vad.ipynb`)
> Standardized result DTO for the voice-activity-detection task — the data noun VAD tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.vad import (
    TimeRange,
    VADResult
)
```
#### Classes

```python
@dataclass
class TimeRange:
    "A temporal segment within an audio source (the VAD speech/silence span)."
    
    start: float  # Start time in seconds
    end: float  # End time in seconds
    label: str = 'speech'  # Segment type (e.g. 'speech')
    confidence: Optional[float]  # Detection confidence (0.0 to 1.0)
    payload: Dict[str, Any] = field(...)  # Extra data (reserved)
    
    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        "Convert to dictionary for JSON serialization."
```

```python
@dataclass
class VADResult:
    "Standardized output for voice-activity-detection capabilities."
    
    ranges: List[TimeRange]  # Detected speech segments, sorted by start
    metadata: Dict[str, Any] = field(...)  # Global VAD stats (duration, sample_rate, total_speech, ...)
    
    def from_dict(
        "Reconstruct from a wire payload, re-typing nested TimeRanges.

`ranges` holds typed `TimeRange` objects, so the substrate's typed wire
envelope (stage 2) reconstructs them host-side here rather than leaving
bare dicts (which would break attribute access like `r.start`)."
```
